# California Housing Price Regression with ANN

## Objectives
Predict **median house value** (regression) with a feedforward network.

## Theory
**Regression output:** single neuron with linear activation.  
**Loss:** Mean Squared Error (MSE) — penalizes large errors quadratically.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Business Context
Lenders and investors estimate property values; regression ANNs capture non-linear feature interactions.


In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(df.head())


In [ ]:
sns.histplot(df["MedHouseVal"], kde=True)
plt.title("Target: Median House Value")
plt.show()
sns.pairplot(df[["MedInc", "HouseAge", "MedHouseVal"]], diag_kind="hist", corner=True)
plt.show()


In [ ]:
X = df.drop(columns=["MedHouseVal"]).values.astype(np.float32)
y = df["MedHouseVal"].values.astype(np.float32)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=SEED)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val_s = y_scaler.transform(y_val.reshape(-1, 1)).ravel()
y_test_s = y_scaler.transform(y_test.reshape(-1, 1)).ravel()


In [ ]:
model = models.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="linear"),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()


In [ ]:
cb_reg = [
    callbacks.ModelCheckpoint("ann_housing_best.keras", save_best_only=True, monitor="val_loss"),
    callbacks.EarlyStopping(patience=15, restore_best_weights=True),
]
history = model.fit(
    X_train_s, y_train_s,
    validation_data=(X_val_s, y_val_s),
    epochs=100,
    batch_size=64,
    callbacks=cb_reg,
    verbose=1,
)
pd.DataFrame(history.history)[["loss", "val_loss"]].plot()
plt.show()


In [ ]:
pred_s = model.predict(X_test_s, verbose=0).ravel()
pred = y_scaler.inverse_transform(pred_s.reshape(-1, 1)).ravel()
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
print("MAE:", mean_absolute_error(y_test, pred))
print("R2:", r2_score(y_test, pred))
plt.scatter(y_test, pred, alpha=0.3, s=5)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Housing price: actual vs predicted")
plt.show()


In [ ]:
model.save("ann_housing_final.keras")
import joblib
joblib.dump(scaler, "housing_X_scaler.pkl")
joblib.dump(y_scaler, "housing_y_scaler.pkl")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
